# Bank Customer Term Deposit Subscription Prediction Using Three Classification Models

**Course:** AI Web Development  |  **Assignment:** TASK 06 - Multi-Model Classification and Performance Comparison

## 1. Introduction

**Objective:** Build three classification models on the same dataset, evaluate them with accuracy, precision, recall and confusion matrices, and decide which model performs best.

**Dataset source:** UCI Machine Learning Repository - *Bank Marketing* dataset (file `bank.csv`, about 4,521 records)
https://archive.ics.uci.edu/dataset/222/bank+marketing

**Problem statement:** A bank phones customers to offer a *term deposit*. Using historical campaign data, can we predict whether a customer will subscribe (`yes`) or not (`no`)?

**What is a term deposit?** A savings product where a customer keeps money in the bank for a fixed period at a fixed interest rate.

**Classification type:** Binary classification (only two possible answers: `yes` or `no`).

**Why do banks study campaign results?** Calling customers costs time and money. If past data shows which customers tend to subscribe, the bank can plan future campaigns better.

**Three algorithms:** Logistic Regression, Decision Tree, K-Nearest Neighbors (KNN). Comparing several algorithms on the same data shows which one suits the problem best.

**Expected output:** Predictions for a test set, a performance comparison table, three confusion matrices, and a best-model decision.

> This project demonstrates machine learning. It is not a real banking application.

## 2. Import Libraries

| Library | Purpose |
|---|---|
| `pandas` | Load and handle tables of data (DataFrames) |
| `numpy` | Numerical operations |
| `matplotlib` | Draw and save the confusion matrix images |
| `scikit-learn` | Split data, preprocess, train models, calculate metrics |

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split          # split data into train/test
from sklearn.compose import ColumnTransformer                 # apply different steps to different columns
from sklearn.pipeline import Pipeline                         # chain preprocessing + model
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # encode text columns, scale numbers
from sklearn.linear_model import LogisticRegression           # model 1
from sklearn.tree import DecisionTreeClassifier               # model 2
from sklearn.neighbors import KNeighborsClassifier            # model 3
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

print("Libraries imported successfully")

## 3. Load Dataset

**Step A - Upload `bank.csv` to Colab.** Run the next cell, click **Choose Files**, and select `bank.csv`.
(Alternative: click the folder icon on the left, then the upload icon, and upload the file.)

In [ ]:
from google.colab import files
uploaded = files.upload()

**Step B - Read the CSV.** This file separates columns with a **semicolon (`;`)** instead of a comma.
Pandas expects commas by default, so without `sep=";"` all columns would be squashed into one.

In [ ]:
df = pd.read_csv("bank.csv", sep=";")
df.head()

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
print("Column names:")
print(df.columns.tolist())

In [ ]:
print("Data types:")
print(df.dtypes)
print()
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Target class counts:")
print(df["y"].value_counts())
print()
print("Target class percentages:")
print((df["y"].value_counts(normalize=True) * 100).round(2))

**How to read these outputs**
- `shape` gives (number of rows, number of columns).
- `dtypes` shows `object` (text) columns and `int64` (number) columns.
- `isnull().sum()` counts genuinely empty cells.
- `duplicated().sum()` counts rows that are exact copies of another row.
- The target `y` is usually **imbalanced**: far more `no` than `yes`. This matters when we evaluate models.

## 4. Data Preprocessing

### A. Missing values
A **missing value** is an empty cell (`NaN`). The word **`unknown`** is different: it is a real text value meaning the bank did not record that answer.
We do **not** treat every `unknown` as missing; we keep it as its own category.

In [ ]:
print("Genuine missing values in the whole table:", df.isnull().sum().sum())
print()

text_columns = df.select_dtypes(include="object").columns
unknown_counts = (df[text_columns] == "unknown").sum()
print("Count of the category 'unknown' per text column:")
print(unknown_counts[unknown_counts > 0])

If the first number is 0, there is nothing to fill in. The `unknown` entries stay as a normal category, and the encoder will turn them into their own column.
If genuine missing values did exist, the usual simple fix would be to fill numbers with the median and text with the most frequent value (e.g. `SimpleImputer`).

### B. Duplicates
We remove a row only if it is an exact copy of another row.

In [ ]:
dup_count = df.duplicated().sum()
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Removed", dup_count, "duplicate rows. New shape:", df.shape)
else:
    print("No duplicate rows found - nothing to remove.")

### C. Categorical encoding, D. Scaling, E. Preventing data leakage

- **Encoding:** Models do mathematics, so they need numbers. `OneHotEncoder` turns a column such as `marital` into separate 0/1 columns (one per category).
- **Scaling:** `StandardScaler` rescales numeric columns to have mean 0 and standard deviation 1. **KNN** measures distance between customers; a large-valued column like `balance` would drown out small ones like `campaign` unless we scale. Logistic Regression also trains more smoothly on scaled data.
- **Data leakage:** If we scale or encode using the *whole* dataset, information from the test set leaks into training. So we **split first** and let a `Pipeline` fit the preprocessing on the **training data only**.

Here we only define the preprocessing steps; they get fitted later, inside each model's pipeline.

In [ ]:
# Features = everything except the target 'y' and the leaky 'duration' column
feature_df = df.drop(columns=["y", "duration"])

categorical_cols = feature_df.select_dtypes(include="object").columns.tolist()
numeric_cols = feature_df.select_dtypes(include="number").columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

The same preprocessor is used for all three models so the comparison is fair. (Scaling does not harm the Decision Tree; it simply is not required for it.)

## 5. Features and Target

- `X` = input features (the information we know *before* calling the customer).
- `y` = target: **`yes`** = customer subscribed to the term deposit, **`no`** = customer did not.

**`duration` is excluded.** It is the length of the last call, which is only known *after* the call. Using it to predict the outcome of that call would be **information leakage**.

In [ ]:
X = df.drop(columns=["y", "duration"])
y = df["y"]

# Safety checks
assert "duration" not in X.columns, "duration must not be in the features"
assert "y" not in X.columns, "target must not be in the features"

print("Feature columns used:")
print(X.columns.tolist())
print("X shape:", X.shape, "| y shape:", y.shape)

## 6. Train/Test Split

- **Training data (80%)**: what the model learns from.
- **Testing data (20%)**: new, unseen rows used only to measure performance. If the model saw them during training, the results would be unfairly good.
- `random_state=42` makes the split repeatable.
- `stratify=y` keeps the same `yes`/`no` percentage in both parts.
- All three models use **this same split**, so differences in results come from the models, not from different data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print()
print("Training class percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))
print("Testing class percentages:")
print((y_test.value_counts(normalize=True) * 100).round(2))

## 7. Model 1 - Logistic Regression

**What it is:** A model that calculates a probability (0 to 1) that a customer belongs to the `yes` class, using a weighted sum of the features. If the probability is above 0.5 it predicts `yes`, otherwise `no`. This makes it a natural fit for two-class problems.

**`fit()`** lets the model learn the best weights from the training data. **`predict()`** applies those weights to new rows.

`max_iter=1000` gives the solver enough steps to finish (avoids a convergence warning).

In [ ]:
lr_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
lr_model.fit(X_train, y_train)
print("Logistic Regression trained")

## 8. Model 2 - Decision Tree

**What it is:** A model that asks a series of yes/no questions about the features (for example "is `poutcome` = success?") and follows the branches until it reaches a prediction. Each path from the top to a leaf is a decision rule.

**Difference from Logistic Regression:** Logistic Regression draws one smooth weighted formula; a Decision Tree splits the data into groups using rules.

`max_depth=5` limits the tree to 5 levels of questions so it stays simple and does not memorize the training data.

In [ ]:
dt_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42)),
])
dt_model.fit(X_train, y_train)
print("Decision Tree trained")

## 9. Model 3 - K-Nearest Neighbors (KNN)

**What it is:** To predict a new customer, KNN finds the **K** most similar customers in the training data (the "nearest neighbors") and takes a majority vote of their outcomes.

**K = 5** means the 5 closest training customers vote.

**Why scaling matters:** "Closeness" is a distance. Without scaling, a column with big numbers (like `balance`) would dominate the distance. Our pipeline scales first.

No hyperparameter tuning is done; we simply use K = 5.

In [ ]:
knn_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])
knn_model.fit(X_train, y_train)
print("KNN trained")

## 10. Generate Predictions

`predict()` gives each model's answer for every test row. The pipeline automatically applies the (training-fitted) scaling and encoding to the test data first.

In [ ]:
y_pred_lr = lr_model.predict(X_test)
y_pred_dt = dt_model.predict(X_test)
y_pred_knn = knn_model.predict(X_test)

predictions = pd.DataFrame({
    "Actual Outcome": y_test.values,
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "KNN": y_pred_knn,
})
predictions.head(10)

Each row is one test customer. Compare the actual outcome with each model's prediction; a mismatch is a mistake.
Because most customers are `no`, most of the first rows will probably be `no`.

## 11. Model Evaluation

The positive class is **`yes`** (customer subscribes).

- **Accuracy** = correct predictions / all predictions.
- **Precision** = of the customers the model predicted `yes`, how many really subscribed? (High precision = few wasted calls.)
- **Recall** = of the customers who really subscribed, how many did the model find? (High recall = few missed customers.)

**Confusion matrix terms** (positive = `yes`):
- **True Positive (TP):** predicted `yes`, actually `yes`.
- **True Negative (TN):** predicted `no`, actually `no`.
- **False Positive (FP):** predicted `yes`, actually `no`.
- **False Negative (FN):** predicted `no`, actually `yes` (a missed customer).

**Why accuracy can mislead:** if about 88% of customers are `no`, a model that always says `no` gets about 88% accuracy while finding **zero** subscribers. That is why precision and recall are also needed.

In [ ]:
class_order = ["no", "yes"]   # same order for every model

def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label="yes", zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label="yes", zero_division=0),
    }

results = [
    evaluate("Logistic Regression", y_test, y_pred_lr),
    evaluate("Decision Tree", y_test, y_pred_dt),
    evaluate("KNN", y_test, y_pred_knn),
]

for name, pred in [("Logistic Regression", y_pred_lr), ("Decision Tree", y_pred_dt), ("KNN", y_pred_knn)]:
    cm = confusion_matrix(y_test, pred, labels=class_order)
    print(name)
    print("            Pred no  Pred yes")
    print("Actual no  ", cm[0])
    print("Actual yes ", cm[1])
    print()

Reading the printed matrices: rows are **actual** values, columns are **predicted** values.
Layout: `[[TN, FP], [FN, TP]]`.

## 12. Confusion Matrices (saved as images)

In [ ]:
os.makedirs("confusion_matrices", exist_ok=True)

matrix_info = [
    ("Logistic Regression", y_pred_lr, "confusion_matrix_logistic_regression.png"),
    ("Decision Tree", y_pred_dt, "confusion_matrix_decision_tree.png"),
    ("KNN", y_pred_knn, "confusion_matrix_knn.png"),
]

for name, pred, filename in matrix_info:
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=class_order, display_labels=class_order,
        cmap="Blues", values_format="d"
    )
    disp.ax_.set_title("Confusion Matrix - " + name)
    disp.ax_.set_xlabel("Predicted label")
    disp.ax_.set_ylabel("Actual label")
    disp.figure_.savefig(os.path.join("confusion_matrices", filename), dpi=150, bbox_inches="tight")
    plt.show()

print("Saved:", os.listdir("confusion_matrices"))

**How to interpret each matrix:** the top-left cell (TN) and bottom-right cell (TP) are correct predictions; the other two cells are mistakes. A good model has large numbers on that diagonal. Pay special attention to the bottom row: how many real `yes` customers were found (TP) versus missed (FN).

## 13. Model Performance Comparison

In [ ]:
comparison = pd.DataFrame(results).round(4)
comparison.to_csv("model_comparison.csv", index=False)
comparison

## 14. Best Model Selection

**Selection rule used in this project:** The classes are imbalanced, so accuracy alone is not enough. We give the most weight to **recall for `yes`** (finding customers who would subscribe), and then check **precision** and **accuracy** so that we do not choose a model that just says `yes` to everyone. Ranking order: Recall, then Precision, then Accuracy.

In [ ]:
print("Highest accuracy :", comparison.loc[comparison["Accuracy"].idxmax(), "Model"], comparison["Accuracy"].max())
print("Highest precision:", comparison.loc[comparison["Precision"].idxmax(), "Model"], comparison["Precision"].max())
print("Highest recall   :", comparison.loc[comparison["Recall"].idxmax(), "Model"], comparison["Recall"].max())
print()

ranked = comparison.sort_values(by=["Recall", "Precision", "Accuracy"], ascending=False).reset_index(drop=True)
print("Ranking by the selection rule:")
print(ranked)
print()
print("Best model by the rule:", ranked.loc[0, "Model"])

In [ ]:
# Extra check: how many customers did each model predict as 'yes'? (actual 'yes' in test set shown too)
print("Actual yes in test set:", (y_test == "yes").sum())
print("Predicted yes - Logistic Regression:", (y_pred_lr == "yes").sum())
print("Predicted yes - Decision Tree      :", (y_pred_dt == "yes").sum())
print("Predicted yes - KNN                :", (y_pred_knn == "yes").sum())

**Your analysis (write this after running the cells above, using your own numbers):**

- Highest accuracy: `[model, value]`
- Highest precision: `[model, value]`
- Highest recall: `[model, value]`
- Differences: `[e.g. one model finds more real subscribers but makes more wrong "yes" predictions]`
- Best overall model: `[model]` because `[its recall for yes compared with the others, and what its precision and accuracy show]`.
- If no model clearly wins, write the trade-off instead of forcing a winner (for example: "Model A has the best accuracy but misses most subscribers; Model B finds more subscribers at the cost of more false alarms").

## 15. Conclusion

**Write the final conclusion after running the notebook**, using your actual results. Include:

- Dataset: UCI Bank Marketing (`bank.csv`), `[number]` records, target `y` (`yes`/`no`).
- Models: Logistic Regression, Decision Tree (max_depth=5), KNN (K=5).
- Preprocessing: checked missing values and duplicates, kept `unknown` as a category, one-hot encoded categorical columns, scaled numeric columns, removed `duration` and `y` from features, split 80/20 with `stratify=y`, fitted preprocessing on training data only.
- Results: `[summarise the comparison table]`.
- Best model: `[name]`, because `[reason based on recall, precision and accuracy]`.

In [ ]:
# Prints the comparison table in Markdown so you can paste it into Classification_Report.md
print("| Model | Accuracy | Precision | Recall |")
print("|---|---|---|---|")
for _, row in comparison.iterrows():
    print("| " + row["Model"] + " | " + format(row["Accuracy"], ".4f") + " | " + format(row["Precision"], ".4f") + " | " + format(row["Recall"], ".4f") + " |")

In [ ]:
# Optional: download the output files from Colab to your computer
from google.colab import files
files.download("model_comparison.csv")
for f in os.listdir("confusion_matrices"):
    files.download(os.path.join("confusion_matrices", f))